In [54]:
import os
import json
from pathlib import Path
from typing import Optional, List, Literal, Dict, Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [36]:
# 当前路径
BASE_DIR = Path.cwd()

# 自动找到 project root
if (BASE_DIR / ".env").exists():
    env_path = BASE_DIR / ".env"
else:
    env_path = BASE_DIR.parent / ".env"

# 加载
load_dotenv(env_path)

# 读取
api_key = os.getenv("OPENAI_API_KEY")

print("API KEY:", api_key[:10], "...")

API KEY: sk-proj-QH ...


In [41]:
# 初始化llm
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

## **Building Langchain Input Layer**

In [46]:
# =========================
# ✅ 改进版 Input Layer（更鲁棒 + 可扩展）
# =========================

# 你当前版本其实已经很好了，我帮你做的是：
# 1. 增强可扩展性（future-proof）
# 2. 加入 routing-ready 字段
# 3. 提升真实业务场景的容错能力


# =========================
# ⭐ 关键升级点
# =========================

# 1. 增加 intent_confidence
# 2. 增加 multi_intent 支持
# 3. 增加 query_type（question / command / unclear）
# 4. 增加 risk_level（是否需要人工介入）
# 5. 增加 retrieval_hints（为RAG优化）
# 6. 增加 tool_hints（未来接tools用）


# =========================
# Context（增强版）
# =========================
from typing import List, Optional, Literal
from pydantic import BaseModel, Field


# =========================
# 1. Shared literal types
# =========================

PrimaryIntent = Literal[
    "product_inquiry",
    "technical_question",
    "pricing_question",
    "timeline_question",
    "customization_request",
    "documentation_request",
    "shipping_question",
    "troubleshooting",
    "order_support",
    "complaint",
    "follow_up",
    "partnership_request",
    "general_info",
    "unknown",
]

LanguageType = Literal["zh", "en", "other"]
ChannelType = Literal["internal_qa", "email", "chat", "unknown"]
QueryType = Literal["question", "request", "unclear"]
UrgencyType = Literal["low", "medium", "high"]
RiskLevelType = Literal["low", "medium", "high"]


# =========================
# 2. Strongly typed schema blocks
# =========================

# 用于分析对话的意图
# basemodel是pydantic
class ParsedContext(BaseModel):
    language: LanguageType = "other"
    channel: ChannelType = "internal_qa"

    primary_intent: PrimaryIntent = "unknown" # 主意图
    secondary_intents: List[PrimaryIntent] = Field(default_factory=list)

    # AI 对自己判断的把握。如果低于 0.5，系统可以自动触发追问逻辑，而不是盲目执行。
    intent_confidence: float = Field(default=0.0, ge=0.0, le=1.0)

    query_type: QueryType = "question"
    urgency: UrgencyType = "low"
    risk_level: RiskLevelType = "low" # 安全阀

    needs_human_review: bool = False # 人工介入与否 属于安全阀
    reasoning_note: str = ""

# 信息的“集装箱” 负责收集对话中都提及了什么信息
class Entities(BaseModel):
    product_names: List[str] = Field(default_factory=list)
    catalog_numbers: List[str] = Field(default_factory=list)
    service_names: List[str] = Field(default_factory=list)
    targets: List[str] = Field(default_factory=list)
    species: List[str] = Field(default_factory=list)
    applications: List[str] = Field(default_factory=list)
    order_numbers: List[str] = Field(default_factory=list)
    document_names: List[str] = Field(default_factory=list)
    company_names: List[str] = Field(default_factory=list)

# 逻辑触发器 
class RequestFlags(BaseModel):
    needs_price: bool = False
    needs_timeline: bool = False
    needs_protocol: bool = False
    needs_customization: bool = False
    needs_order_status: bool = False
    needs_shipping_info: bool = False
    needs_documentation: bool = False
    needs_troubleshooting: bool = False
    needs_quote: bool = False
    needs_availability: bool = False
    needs_recommendation: bool = False
    needs_comparison: bool = False
    needs_invoice: bool = False
    needs_refund_or_cancellation: bool = False
    needs_sample: bool = False
    needs_regulatory_info: bool = False

# 限制条件 比如用户的预算 时间规划..
class Constraints(BaseModel):
    budget: Optional[str] = None
    timeline_requirement: Optional[str] = None
    destination: Optional[str] = None
    quantity: Optional[str] = None
    grade_or_quality: Optional[str] = None
    usage_context: Optional[str] = None
    format_or_size: Optional[str] = None
    comparison_target: Optional[str] = None
    preferred_supplier_or_brand: Optional[str] = None

# 开放槽位
class OpenSlots(BaseModel):
    customer_goal: Optional[str] = None
    experiment_type: Optional[str] = None
    pain_point: Optional[str] = None
    requested_action: Optional[str] = None
    referenced_prior_context: Optional[str] = None
    delivery_or_logistics_note: Optional[str] = None
    regulatory_or_compliance_note: Optional[str] = None
    other_notes: List[str] = Field(default_factory=list)

# 用于接rag
class RetrievalHints(BaseModel):
    keywords: List[str] = Field(default_factory=list)
    expanded_queries: List[str] = Field(default_factory=list)
    filters: List[str] = Field(default_factory=list)

# 用于接tools
class ToolHints(BaseModel):
    suggested_tools: List[str] = Field(default_factory=list)
    requires_database_lookup: bool = False
    requires_file_lookup: bool = False
    requires_order_system: bool = False

class ParsedResult(BaseModel):
    normalized_query: str = ""

    context: ParsedContext = Field(default_factory=ParsedContext)
    entities: Entities = Field(default_factory=Entities)
    request_flags: RequestFlags = Field(default_factory=RequestFlags)
    constraints: Constraints = Field(default_factory=Constraints)
    open_slots: OpenSlots = Field(default_factory=OpenSlots)

    retrieval_hints: RetrievalHints = Field(default_factory=RetrievalHints)
    tool_hints: ToolHints = Field(default_factory=ToolHints)

    missing_information: List[str] = Field(default_factory=list)
    extra_instructions: Optional[str] = None

"""
你的原版已经是：80分（非常强）
这个版本是：95分（接近 production）

核心提升：

1️⃣ 不只是“理解用户” → 还能“指导后续系统”
   - retrieval_hints → 直接优化RAG
   - tool_hints → 直接接订单系统 / CRM

2️⃣ 更真实业务场景
   - intent_confidence → 避免误判
   - risk_level → 控制是否要人工

3️⃣ 支持复杂用户
   - multi-intent
   - 模糊 query

4️⃣ 直接为 routing 做准备
   - 可以做：
     if primary_intent == "pricing": → pricing chain
     if tool_hints.requires_order_system: → 调数据库
"""

'\n你的原版已经是：80分（非常强）\n这个版本是：95分（接近 production）\n\n核心提升：\n\n1️⃣ 不只是“理解用户” → 还能“指导后续系统”\n   - retrieval_hints → 直接优化RAG\n   - tool_hints → 直接接订单系统 / CRM\n\n2️⃣ 更真实业务场景\n   - intent_confidence → 避免误判\n   - risk_level → 控制是否要人工\n\n3️⃣ 支持复杂用户\n   - multi-intent\n   - 模糊 query\n\n4️⃣ 直接为 routing 做准备\n   - 可以做：\n     if primary_intent == "pricing": → pricing chain\n     if tool_hints.requires_order_system: → 调数据库\n'

In [47]:
# 不变量
# 包含了所有的业务规则、意图定义、实体提取指南。
# 无论用户问什么，这段规则都不会变
PARSER_SYSTEM_PROMPT = """
You are an input parser for a biotech customer-support AI agent.

Your job is to convert a user's natural-language message into structured data for downstream routing, retrieval, and response drafting.
You are NOT answering the user directly.
You are NOT writing an email reply.
You are only extracting structured information as accurately and conservatively as possible.

General rules:
1. Extract only information that is explicitly stated or strongly implied.
2. Do not invent product names, catalog numbers, prices, timelines, shipping details, protocols, or scientific claims.
3. If a field is not clearly supported by the user input, leave it empty, false, unknown, or null depending on the schema.
4. Be conservative. When uncertain, prefer leaving fields empty rather than guessing.
5. Choose one primary intent and optionally multiple secondary intents.
6. Use missing_information to note key details that may be required for downstream handling.
7. Set needs_human_review=true if the message is high-risk, escalated, sensitive, complaint-heavy, or likely requires human judgment.
8. Keep reasoning_note short, factual, and non-speculative.

How to interpret the schema:
- normalized_query: a cleaned version of the user's request that preserves meaning while removing obvious noise
- context: the overall meaning, tone, urgency, and handling risk of the message
- entities: concrete things explicitly mentioned, such as products, services, targets, species, documents, order numbers, or company names
- request_flags: what the user is asking for or needs help with
- constraints: hard or soft requirements that affect retrieval, recommendation, or action
- open_slots: useful contextual information that does not fit neatly into fixed schema fields
- missing_information: important details that are not provided but may be needed later
- extra_instructions: any downstream handling note that is directly supported by the message

Intent guidance:
- product_inquiry: asks whether a product/service exists, is available, or requests general product details
- technical_question: asks about scientific rationale, protocol, experimental design, validation, mechanism, or technical suitability
- pricing_question: asks about price, cost, discount, quotation, or budget-related matters
- timeline_question: asks about turnaround time, lead time, ETA, or delivery timing
- customization_request: asks for a custom design, custom service, modification, or tailored solution
- documentation_request: asks for datasheet, protocol, brochure, COA, SDS, validation file, or technical documentation
- shipping_question: asks about shipping method, destination, delivery, customs, tracking, or logistics
- troubleshooting: asks why something failed, how to fix it, or how to optimize a technical issue
- order_support: asks about an existing order, invoice, payment, PO, order changes, order status, or cancellation
- complaint: expresses dissatisfaction, blame, refund demand, service failure, or escalation
- follow_up: asks for an update on a previous quote, order, email thread, experiment, or prior request
- partnership_request: asks about collaboration, distributorship, partnership, or business cooperation
- general_info: broad or general company/service information request
- unknown: the message is too ambiguous to classify reliably

Entity extraction guidance:
Extract only entities that are mentioned in the message.
Possible entity types include:
- product_names
- catalog_numbers
- service_names
- targets
- species
- applications
- order_numbers
- document_names
- company_names

Request flag guidance:
Turn user needs into boolean signals when supported by the message.
Examples:
- needs_price: asking about price or cost
- needs_timeline: asking how long something takes or when it can arrive
- needs_protocol: asking for protocol, workflow, process, method, or experiment steps
- needs_customization: asking whether something can be customized or specially designed
- needs_order_status: asking for progress/status of an existing order
- needs_shipping_info: asking about shipping/tracking/customs/delivery
- needs_documentation: asking for datasheet, brochure, COA, SDS, manual, technical file
- needs_troubleshooting: asking how to solve a technical or product-use problem
- needs_quote: explicitly asking for a quotation
- needs_availability: asking whether the product/service exists, is available, or is offered
- needs_recommendation: asking which option is better, best, or most suitable
- needs_comparison: asking to compare products, services, formats, or solutions
- needs_invoice: asking for invoice, PO, payment paperwork, billing details
- needs_refund_or_cancellation: asking to cancel, return, refund, or revise an order
- needs_sample: asking for sample, trial material, evaluation unit
- needs_regulatory_info: asking for compliance, import/export documents, certificates, or regulatory details

Constraint guidance:
Extract only if explicit or strongly implied:
- budget
- timeline_requirement
- destination
- quantity
- grade_or_quality
- usage_context
- format_or_size
- comparison_target
- preferred_supplier_or_brand

Open slot guidance:
Use open_slots for important context that does not fit fixed schema cleanly:
- customer_goal
- experiment_type
- pain_point
- requested_action
- referenced_prior_context
- delivery_or_logistics_note
- regulatory_or_compliance_note
- other_notes

Language guidance:
- zh for Chinese
- en for English
- other otherwise

Channel guidance:
- email if the message clearly looks like an email
- chat if it is short conversational messaging
- internal_qa otherwise

Human review guidance:
Set needs_human_review=true if any of the following apply:
- strong complaint or escalation
- refund dispute or order dispute
- legal/compliance risk
- high-stakes technical recommendation without enough information
- the user requests guarantees or commitments not supported by available facts
- the message is too ambiguous, emotionally charged, or risky for fully automated handling

Return only structured data matching the schema.
""".strip()

In [48]:
# LangChain 收到 LLM 吐出的 JSON 字典，自动把它变成了一个 Pydantic 对象
structured_llm = llm.with_structured_output(ParsedResult)

# 可变对象 由ChatPromptTemplate.from_messages 创建出来的 代码对象
# 它不仅包含了上面的 PARSER_SYSTEM_PROMPT（作为 system 角色），还包含了一个 human 角色的模板，里面留好了 {user_query} 这样的占位符
parser_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", PARSER_SYSTEM_PROMPT), # 使用 ChatPromptTemplate 将 PARSER_SYSTEM_PROMPT（灵魂）和用户输入（原材料）分离开，这非常利于调试和维护。
        (
            "human",
            """
Parse the following user input into the target schema.

User query:
{user_query}

Conversation history:
{conversation_history}

Attachments:
{attachments}
""".strip(),
        ),
    ]
)

# ｜是langchain专用的符号 表示一个管道
# 在这里就是说按照我们的system prompt把复杂文本喂给structured llm
# 输出parsed result对象
parser_chain = parser_prompt | structured_llm

In [49]:
# 解析
# 将自然语言转换为ParsedResult
# 启动 LLM 链条进行“深度理解”，用AI对原始文本进行理解
def parse_user_input(
    user_query: str,
    conversation_history: Optional[List[Dict[str, str]]] = None,
    attachments: Optional[List[Dict[str, Any]]] = None,
) -> ParsedResult:
    conversation_history = conversation_history or []
    attachments = attachments or []

    parsed = parser_chain.invoke(
        {
            "user_query": user_query,
            "conversation_history": json.dumps(conversation_history, ensure_ascii=False, indent=2),
            "attachments": json.dumps(attachments, ensure_ascii=False, indent=2),
        }
    )

    return parsed

# 包装
# 将“AI 解析后的结果”和“原始环境数据”合二为一
# 转换成agent_input
def build_agent_input(
    original_query: str, # 原始文本
    parsed: ParsedResult,
    conversation_history: Optional[List[Dict[str, str]]] = None, # 提供对话记录，上下文，给下游 Agent 提供前因后果
    attachments: Optional[List[Dict[str, Any]]] = None, # 如果用户发了图片或 PDF，Agent 需要知道这些文件的存在
) -> Dict[str, Any]:
    conversation_history = conversation_history or []
    attachments = attachments or []

    return {
        "query": parsed.normalized_query or original_query.strip(),
        "context": parsed.context.model_dump(),
        "entities": parsed.entities.model_dump(),
        "request_flags": parsed.request_flags.model_dump(),
        "constraints": parsed.constraints.model_dump(),
        "open_slots": parsed.open_slots.model_dump(),
        "retrieval_hints": parsed.retrieval_hints.model_dump(),
        "tool_hints": parsed.tool_hints.model_dump(),
        "missing_information": parsed.missing_information,
        "conversation_history": conversation_history,
        "attachments": attachments,
        "extra_instructions": parsed.extra_instructions,
    }

# 封装以上两步的函数
def make_agent_input(
    user_query: str,
    conversation_history: Optional[List[Dict[str, str]]] = None,
    attachments: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    parsed = parse_user_input(
        user_query=user_query,
        conversation_history=conversation_history,
        attachments=attachments,
    )

    return build_agent_input(
        original_query=user_query,
        parsed=parsed,
        conversation_history=conversation_history,
        attachments=attachments,
    )

## **Testing**

In [51]:
def test_single_query(query: str):
    print("=" * 80)
    print("Query:", query)

    parsed = parse_user_input(query)

    print("\n[ParsedResult]")
    print(json.dumps(parsed.model_dump(), indent=2, ensure_ascii=False))

    agent_input = build_agent_input(query, parsed)

    print("\n[Agent Input]")
    print(json.dumps(agent_input, indent=2, ensure_ascii=False))

In [52]:
test_single_query(
    "Do you have CD19 CAR-T products for in vivo mouse study?"
)

Query: Do you have CD19 CAR-T products for in vivo mouse study?

[ParsedResult]
{
  "normalized_query": "Do you have CD19 CAR-T products for in vivo mouse study?",
  "context": {
    "language": "en",
    "channel": "unknown",
    "primary_intent": "product_inquiry",
    "secondary_intents": [],
    "intent_confidence": 0.9,
    "query_type": "question",
    "urgency": "low",
    "risk_level": "low",
    "needs_human_review": false,
    "reasoning_note": "User is inquiring about the availability of a specific product."
  },
  "entities": {
    "product_names": [
      "CD19 CAR-T products"
    ],
    "catalog_numbers": [],
    "service_names": [],
    "targets": [],
    "species": [
      "mouse"
    ],
    "applications": [
      "in vivo study"
    ],
    "order_numbers": [],
    "document_names": [],
    "company_names": []
  },
  "request_flags": {
    "needs_price": false,
    "needs_timeline": false,
    "needs_protocol": false,
    "needs_customization": false,
    "needs_order_

In [53]:
test_single_query(
    "Can you send me a quote and datasheet for PM-CAR1067 by this week?"
)

Query: Can you send me a quote and datasheet for PM-CAR1067 by this week?

[ParsedResult]
{
  "normalized_query": "Can you send me a quote and datasheet for PM-CAR1067 by this week?",
  "context": {
    "language": "en",
    "channel": "unknown",
    "primary_intent": "pricing_question",
    "secondary_intents": [
      "documentation_request"
    ],
    "intent_confidence": 0.9,
    "query_type": "request",
    "urgency": "medium",
    "risk_level": "low",
    "needs_human_review": false,
    "reasoning_note": "User is requesting a quote and datasheet for a specific product."
  },
  "entities": {
    "product_names": [
      "PM-CAR1067"
    ],
    "catalog_numbers": [],
    "service_names": [],
    "targets": [],
    "species": [],
    "applications": [],
    "order_numbers": [],
    "document_names": [
      "datasheet"
    ],
    "company_names": []
  },
  "request_flags": {
    "needs_price": true,
    "needs_timeline": true,
    "needs_protocol": false,
    "needs_customization":